# Ćwiczenie 2 — NumPy (Add-on): Algorytmy sortowania od Bubble Sort do Quick Sorta

Ten notebook jest **osobnym, dydaktycznym dodatkiem** do sekcji `02.08 Sorting`.

Nie chodzi tu o to, żeby pisać szybszy kod niż `sorted()` albo `np.sort`.  
Chodzi o to, żeby zobaczyć **skąd bierze się różnica między prostymi algorytmami O(n^2) a algorytmami O(n log n)**, jaką rolę gra **stabilność**, kiedy pomaga **rekurencja**, i dlaczego niektóre algorytmy są dobre tylko w bardzo konkretnych sytuacjach.


## Cel notebooka

Po ukończeniu tego zestawu student powinien:
1. Umieć zaimplementować i opisać:
   - **bubble sort**,
   - **selection sort**,
   - **insertion sort**,
   - **merge sort**,
   - **quicksort**.
2. Rozumieć różnicę:
   - **stabilny vs niestabilny**,
   - **in-place vs out-of-place**,
   - **iteracyjny vs rekurencyjny**.
3. Umieć wyjaśnić, **dlaczego sama rekurencja nie przyspiesza kodu**, ale dobrze zapisuje algorytmy typu **divide-and-conquer**.
4. Umieć porównać zachowanie algorytmów na danych:
   - losowych,
   - prawie posortowanych,
   - odwróconych.


## Jak pracować z notebookiem

- W komórkach z `# TODO` uzupełnij kod.
- Po każdej implementacji uruchom komórkę z testami `assert`.
- Tam, gdzie pojawia się pseudokod, postaraj się najpierw "przetłumaczyć" go na Python w głowie.
- Wersja `done` pokazuje jedno z poprawnych rozwiązań, ale **nie jedyne możliwe**.


In [ ]:
from __future__ import annotations

from typing import Callable, List, Sequence, TypeVar, Optional
import random
import numpy as np

from cwiczenie_2_numpy_sortowanie_algorytmy_core import (
    RANDOM_SEED,
    is_sorted,
    same_multiset,
    time_call,
    make_random_list,
    make_nearly_sorted_list,
    make_reversed_list,
    benchmark_suite,
    print_benchmark_table,
)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

T = TypeVar("T")

np.set_printoptions(edgeitems=6, linewidth=120)


## 0. Po co implementować sort ręcznie, skoro mamy `sorted()` i `np.sort`?

Bo ręczna implementacja odsłania rzeczy, których nie widać z poziomu gotowej funkcji:

- **bubble / selection / insertion** pokazują, jak lokalne reguły porządkowania przekładają się na koszt `O(n^2)`,
- **merge sort** pokazuje, jak działa **divide-and-conquer**,
- **quicksort** pokazuje siłę dobrego **partycjonowania**,
- **stabilność** staje się namacalna dopiero wtedy, gdy sami piszemy porównania i testy.

W praktyce:
- dla list używamy zwykle `sorted()` / `list.sort()`,
- dla tablic NumPy używamy `np.sort()` / `np.argsort()`.

Ten notebook ma więc charakter **algorytmiczny**, a nie "produkcyjny".


## 0.1 Rekurencja: co naprawdę "przyspiesza"?

Bardzo ważna uwaga dydaktyczna:

> **Rekurencja sama w sobie nie przyspiesza programu.**

Często jest wręcz trochę wolniejsza od pętli, bo każde wywołanie funkcji ma swój koszt.  
To, co daje przyspieszenie, to **pomysł algorytmiczny**.

### Jak to wygląda w merge sort?
- Dzielimy problem na dwie połowy.
- Rekurencyjnie sortujemy każdą połowę.
- Scalanie dwóch posortowanych połówek robimy w czasie liniowym.

Na każdym poziomie "drzewa" rekurencji dotykamy łącznie około `n` elementów, a poziomów jest około `log2(n)`, więc łącznie dostajemy `O(n log n)`.

### Jak to wygląda w quicksort?
- Jednym przejściem **partycjonujemy** dane względem pivota.
- Potem rekurencyjnie sortujemy lewą i prawą część.
- Jeśli pivot dzieli dane w miarę równo, znowu dostajemy około `O(n log n)`.

### Czyli "rekurencja = szybko"?
Nie.  
Raczej:

- **rekurencja** jest wygodnym sposobem zapisania algorytmu,
- **divide-and-conquer** daje lepszą złożoność,
- **lepsza złożoność** daje przewagę dla dużych `n`.


## 0.2 Ściąga: stabilność, pamięć, typowe zastosowania

| Algorytm | Średni czas | Najgorszy czas | Dodatkowa pamięć | Stabilny? | Dobre zastosowanie dydaktyczne |
|---|---:|---:|---:|---|---|
| Bubble sort | O(n²) | O(n²) | O(1) | tak | zrozumienie lokalnych zamian |
| Selection sort | O(n²) | O(n²) | O(1) | zwykle nie | zrozumienie "wybierania minimum" |
| Insertion sort | O(n²) | O(n²) | O(1) | tak | małe lub prawie posortowane dane |
| Merge sort | O(n log n) | O(n log n) | O(n) | tak | stabilność + divide-and-conquer |
| Quicksort | O(n log n) | O(n²) | O(log n) rekurencji | zwykle nie | szybkie sortowanie in-place na danych losowych |
| `sorted()` / Timsort | bardzo dobre w praktyce | O(n log n) | zależne od implementacji | tak | praktyka produkcyjna |


## 1. Bubble Sort (sortowanie bąbelkowe)

### Intuicja
Przechodzimy po liście i **zamieniamy sąsiadów**, jeśli są w złej kolejności.  
Po jednym pełnym przejściu największy element "wypływa" na koniec - stąd nazwa.

### Co ten algorytm uczy?
- myślenia o **lokalnej poprawie**,
- inwariantu: po każdej rundzie koniec listy jest już poprawny,
- znaczenia warunku "czy była jakaś zamiana?".

### Kiedy ma sens?
Głównie dydaktycznie.  
W praktyce prawie nigdy go nie wybieramy, ale jest świetny jako pierwszy krok do zrozumienia sortowania.

### Pseudokod
```text
for end = n-1 down to 1:
    swapped = False
    for i = 0 .. end-1:
        if a[i] > a[i+1]:
            swap(a[i], a[i+1])
            swapped = True
    if not swapped:
        break
```

### Stabilność
Jeśli zamieniasz elementy tylko wtedy, gdy `left > right` (a nie `>=`), to algorytm jest **stabilny**.


In [ ]:
def bubble_sort(arr: Sequence[T], key: Callable[[T], object] = lambda x: x) -> List[T]:
    """Return a sorted copy of arr using bubble sort.

    Requirements:
    - MUST NOT modify the input.
    - SHOULD be stable.
    - Return a Python list.
    """
    a = list(arr)

    # PSEUDOKOD:
    # 1) skopiuj wejście do listy a
    # 2) ustaw koniec aktywnego fragmentu od n-1 do 1
    # 3) w każdej rundzie przejdź po parach sąsiadów:
    #       jeśli key(a[i]) > key(a[i+1]), zamień je miejscami
    #       zapamiętaj, że była zamiana
    # 4) jeśli w całej rundzie nie było zamiany, przerwij wcześniej
    # 5) zwróć a

    # TODO: implement bubble sort
    raise NotImplementedError


In [ ]:
# Tests — bubble sort
def _test_bubble_sort():
    for n in [0, 1, 2, 5, 20]:
        for _ in range(20):
            xs = [random.randint(-10, 10) for _ in range(n)]
            out = bubble_sort(xs)
            assert out == sorted(xs), f"Mismatch: {xs} -> {out}"

    records = [("A", 2), ("B", 2), ("C", 1), ("D", 2)]
    out = bubble_sort(records, key=lambda r: r[1])
    assert out == [("C", 1), ("A", 2), ("B", 2), ("D", 2)], out
    print("✅ bubble_sort passed")

_test_bubble_sort()


## 2. Selection Sort (sortowanie przez wybieranie minimum)

### Intuicja
Na pozycji `i` chcemy położyć **najmniejszy** element z fragmentu `a[i:]`.

Czyli:
- w pierwszym kroku szukamy minimum całej listy,
- w drugim kroku minimum z ogona,
- itd.

### Co ten algorytm uczy?
- jak działa pomysł "wybierz minimum z reszty",
- dlaczego **mało zamian** nie oznacza jeszcze, że algorytm będzie szybki,
- dlaczego prostota nie gwarantuje stabilności.

### Kiedy bywa sensowny?
Teoretycznie bywa rozważany wtedy, gdy **zamiana elementów jest bardzo droga**, bo wykonuje ich mało.  
Ale nadal wykonuje dużo porównań: około `n²/2`.

### Pseudokod
```text
for i = 0 .. n-1:
    min_idx = i
    for j = i+1 .. n-1:
        if a[j] < a[min_idx]:
            min_idx = j
    swap(a[i], a[min_idx])
```

### Stabilność
Klasyczna wersja jest zwykle **niestabilna**, bo końcowa zamiana może przeskoczyć przez elementy o równych kluczach.


In [ ]:
def selection_sort(arr: Sequence[T], key: Callable[[T], object] = lambda x: x) -> List[T]:
    """Return a sorted copy of arr using classic selection sort.

    Requirements:
    - MUST NOT modify the input.
    - Return a Python list.
    - This classic version does NOT need to be stable.
    """
    a = list(arr)

    # PSEUDOKOD:
    # 1) dla każdej pozycji i:
    #       załóż, że minimum jest na i
    # 2) przejrzyj ogon a[i+1:]
    #       jeśli znajdziesz mniejszy klucz, zapamiętaj jego indeks
    # 3) po zakończeniu wewnętrznej pętli zamień a[i] z a[min_idx]
    # 4) po ostatniej iteracji zwróć a

    # TODO: implement selection sort
    raise NotImplementedError


In [ ]:
# Tests — selection sort
def _test_selection_sort():
    for n in [0, 1, 2, 5, 20]:
        for _ in range(20):
            xs = [random.randint(-10, 10) for _ in range(n)]
            out = selection_sort(xs)
            assert is_sorted(out)
            assert same_multiset(xs, out)

    # Correctness only: classic selection sort is usually NOT stable.
    records = [("A", 2), ("B", 2), ("C", 1)]
    out = selection_sort(records, key=lambda r: r[1])
    assert is_sorted(out, key=lambda r: r[1])
    assert same_multiset(records, out)
    print("✅ selection_sort passed")

_test_selection_sort()

records = [("A", 2), ("B", 2), ("C", 1), ("D", 2)]
print("Przykład stabilności / niestabilności:")
print("wejście :", records)
print("wyjście :", selection_sort(records, key=lambda r: r[1]))


## 3. Insertion Sort (sortowanie przez wstawianie)

### Intuicja
Budujemy **posortowany prefiks** listy:
- dla `i = 1` wstawiamy `a[1]` w dobre miejsce wśród `[a[0]]`,
- dla `i = 2` wstawiamy `a[2]` w dobre miejsce wśród `[a[0], a[1]]`,
- itd.

### Co go wyróżnia?
- jest nadal `O(n^2)` w najgorszym przypadku,
- ale na **prawie posortowanych danych** bywa bardzo szybki,
- i dlatego bywa używany wewnątrz bardziej zaawansowanych algorytmów (np. Timsort dla krótkich fragmentów).

### Dlaczego działa dobrze na prawie posortowanych danych?
Bo każdy element musi wtedy przesunąć się tylko o mało pozycji.  
Koszt zależy więc nie tylko od `n`, ale też od tego, **jak bardzo dane są "nieuporządkowane"**.

### Pseudokod
```text
for i = 1 .. n-1:
    x = a[i]
    j = i-1
    while j >= 0 and a[j] > x:
        a[j+1] = a[j]
        j--
    a[j+1] = x
```

### Stabilność
Używamy warunku `>` zamiast `>=`.  
Dzięki temu elementy o równym kluczu nie przeskakują przez siebie.


In [ ]:
def insertion_sort(arr: Sequence[T], key: Callable[[T], object] = lambda x: x) -> List[T]:
    """Return a sorted copy of arr using insertion sort.

    Requirements:
    - MUST NOT modify the input sequence.
    - MUST be stable with respect to key.
    - Return a Python list.
    """
    a = list(arr)

    # PSEUDOKOD:
    # 1) dla i od 1 do n-1:
    #       zapamiętaj aktualny element x = a[i]
    #       zapamiętaj jego klucz xk = key(x)
    # 2) idź wskaźnikiem j w lewo przez posortowany prefiks
    # 3) dopóki key(a[j]) > xk:
    #       przesuwaj a[j] o jedno miejsce w prawo
    # 4) wstaw x na pozycję j+1
    #
    # UWAGA: użyj '>' zamiast '>=' aby zachować stabilność.

    # TODO: implement insertion sort
    raise NotImplementedError


In [ ]:
# Tests — insertion sort
def _test_insertion_sort():
    for n in [0, 1, 2, 5, 20]:
        for _ in range(20):
            xs = [random.randint(-10, 10) for _ in range(n)]
            out = insertion_sort(xs)
            assert out == sorted(xs), f"Mismatch: {xs} -> {out}"

    records = [("A", 2), ("B", 2), ("C", 1), ("D", 2)]
    out = insertion_sort(records, key=lambda r: r[1])
    assert out == [("C", 1), ("A", 2), ("B", 2), ("D", 2)], out
    print("✅ insertion_sort passed")

_test_insertion_sort()


## 4. Merge Sort (sortowanie przez scalanie)

### Najpierw intuicja
Merge sort robi dwie rzeczy:

1. **dzieli** listę na coraz mniejsze fragmenty, aż do list długości 0 lub 1,
2. **scala** posortowane fragmenty z powrotem w większe listy.

### Mini-ślad dla `[5, 2, 4, 1]`
- dzielimy na `[5, 2]` i `[4, 1]`
- dalej na `[5]`, `[2]`, `[4]`, `[1]`
- scalamy `[5] + [2] -> [2, 5]`
- scalamy `[4] + [1] -> [1, 4]`
- scalamy `[2, 5] + [1, 4] -> [1, 2, 4, 5]`

### Dlaczego to jest szybkie?
Bo "drogi" fragment pracy nie polega na chaotycznym wielokrotnym przesuwaniu elementów, tylko na **kontrolowanym scalaniu dwóch już posortowanych list**.

### Co dokładnie daje złożoność `O(n log n)`?
- poziomów podziału jest około `log2(n)`,
- na każdym poziomie scalamy łącznie `n` elementów,
- więc całkowity koszt to około `n * log2(n)`.

### Cena
Merge sort potrzebuje dodatkowej pamięci na listy pomocnicze.  
To jest klasyczny kompromis: **przewidywalny czas za cenę większej pamięci**.


In [ ]:
def merge(left: Sequence[T], right: Sequence[T], key: Callable[[T], object] = lambda x: x) -> List[T]:
    """Merge two already-sorted sequences into one sorted list (stable)."""
    out: List[T] = []
    i = 0
    j = 0

    # PSEUDOKOD:
    # 1) miej dwa wskaźniki i, j ustawione na początek left i right
    # 2) dopóki obie listy mają jeszcze elementy:
    #       jeśli key(left[i]) <= key(right[j]):
    #           dopisz left[i] i przesuń i
    #       w przeciwnym razie:
    #           dopisz right[j] i przesuń j
    # 3) na końcu dopisz resztę niewykorzystanej listy
    #
    # UWAGA: przy równości bierz z lewej, aby zachować stabilność.

    # TODO: implement stable merge
    raise NotImplementedError


def merge_sort(arr: Sequence[T], key: Callable[[T], object] = lambda x: x) -> List[T]:
    """Return a sorted copy of arr using merge sort (stable)."""
    a = list(arr)

    # PSEUDOKOD:
    # 1) jeśli długość a <= 1, zwróć a
    # 2) wyznacz środek mid
    # 3) rekurencyjnie posortuj lewą połowę
    # 4) rekurencyjnie posortuj prawą połowę
    # 5) zwróć merge(left, right)

    # TODO: implement merge sort
    raise NotImplementedError


In [ ]:
# Tests — merge sort
def _test_merge_sort():
    for n in [0, 1, 2, 5, 50]:
        for _ in range(20):
            xs = [random.randint(-50, 50) for _ in range(n)]
            out = merge_sort(xs)
            assert out == sorted(xs), f"Mismatch: {xs} -> {out}"

    records = [("A", 2), ("B", 2), ("C", 1), ("D", 2), ("E", 1)]
    out = merge_sort(records, key=lambda r: r[1])
    assert out == [("C", 1), ("E", 1), ("A", 2), ("B", 2), ("D", 2)], out
    print("✅ merge_sort passed")

_test_merge_sort()


## 5. Quick Sort (sortowanie szybkie)

### Intuicja
Wybieramy **pivot** i dzielimy tablicę na dwie części:
- elementy `<= pivot`,
- elementy `> pivot`.

Po jednym kroku partycjonowania pivot trafia na swoje **ostateczne miejsce**.  
Potem wystarczy posortować lewą i prawą część.

### Dlaczego quicksort bywa bardzo szybki?
- pracuje **in-place**,
- ma mały narzut pamięciowy,
- ma dobre stałe ukryte w `O(...)`,
- często dobrze wykorzystuje pamięć podręczną.

### Gdzie jest haczyk?
Jeśli pivot jest źle wybierany (np. zawsze ostatni element), to dla danych już posortowanych albo odwróconych można dostać zły przypadek `O(n^2)`.

### Pseudokod partycjonowania (Lomuto)
```text
pivot = a[hi]
i = lo
for j = lo .. hi-1:
    if a[j] <= pivot:
        swap(a[i], a[j])
        i++
swap(a[i], a[hi])
return i
```

### Rekurencja
Tutaj znowu rekurencja tylko **opisuje strukturę** problemu:
- po partycjonowaniu mamy dwa mniejsze podproblemy,
- każdy rozwiązujemy dokładnie tak samo.


In [ ]:
def partition_lomuto(a: List[T], lo: int, hi: int, key: Callable[[T], object] = lambda x: x) -> int:
    """Partition a[lo:hi+1] around pivot=a[hi]. Return final pivot index."""

    # PSEUDOKOD:
    # 1) pivot = a[hi]
    # 2) i = lo
    # 3) dla j od lo do hi-1:
    #       jeśli key(a[j]) <= key(pivot):
    #           zamień a[i] z a[j]
    #           zwiększ i
    # 4) na końcu zamień a[i] z pivotem a[hi]
    # 5) zwróć i

    # TODO: implement Lomuto partition
    raise NotImplementedError


def quicksort_inplace(a: List[T], lo: int = 0, hi: Optional[int] = None,
                      key: Callable[[T], object] = lambda x: x) -> None:
    """Sort list a in-place using recursive quicksort."""
    if hi is None:
        hi = len(a) - 1

    # PSEUDOKOD:
    # 1) jeśli lo >= hi: nic nie rób
    # 2) znajdź pozycję pivota przez partition_lomuto
    # 3) rekurencyjnie posortuj lewą część
    # 4) rekurencyjnie posortuj prawą część

    # TODO: implement recursive in-place quicksort
    raise NotImplementedError


In [ ]:
# Tests — partition + quicksort
def _test_partition_invariants():
    xs = [3, 7, 2, 9, 1, 5]
    a = xs.copy()
    p = partition_lomuto(a, 0, len(a) - 1)
    pivot = a[p]
    assert all(x <= pivot for x in a[:p]), (a, p)
    assert all(x > pivot for x in a[p+1:]), (a, p)
    assert same_multiset(xs, a)

def _test_quicksort_inplace():
    for n in [0, 1, 2, 10, 50]:
        for _ in range(20):
            xs = [random.randint(-100, 100) for _ in range(n)]
            a = xs.copy()
            quicksort_inplace(a)
            assert is_sorted(a), f"Not sorted: {xs} -> {a}"
            assert same_multiset(xs, a)

_test_partition_invariants()
_test_quicksort_inplace()
print("✅ quicksort passed")

records = [("A", 2), ("B", 2), ("C", 1), ("D", 2)]
a = records.copy()
quicksort_inplace(a, key=lambda r: r[1])
print("Przykład quicksort (stabilność nie jest gwarantowana):")
print("wejście :", records)
print("wyjście :", a)


## 6. Stabilność w praktyce: sortowanie po wielu kluczach

Stabilność nie jest tylko własnością teoretyczną.  
Daje bardzo praktyczny trik:

Aby posortować rekordy po dwóch kluczach:
1. najpierw stabilnie sortujesz po **mniej ważnym** kluczu,
2. potem stabilnie sortujesz po **ważniejszym** kluczu.

Dzięki stabilności drugi sort nie niszczy porządku zbudowanego przez pierwszy.

Właśnie dlatego w bibliotekach tak cenna jest stabilność `sorted()` / `list.sort()`.


In [ ]:
def stable_two_key_sort(
    records: Sequence[T],
    primary_key: Callable[[T], object],
    secondary_key: Callable[[T], object],
) -> List[T]:
    """Stable two-key sort using only a stable one-key sort.

    Requirements:
    - use merge_sort twice,
    - first by secondary_key,
    - then by primary_key.
    """

    # PSEUDOKOD:
    # 1) wykonaj stabilny sort po secondary_key
    # 2) na wyniku wykonaj stabilny sort po primary_key
    # 3) zwróć wynik

    # TODO: implement
    raise NotImplementedError


In [ ]:
# Tests — stable two-key sort
def _test_stable_two_key_sort():
    records = [
        {"name": "A", "city": "WAW", "age": 30},
        {"name": "B", "city": "KRK", "age": 20},
        {"name": "C", "city": "WAW", "age": 20},
        {"name": "D", "city": "KRK", "age": 30},
        {"name": "E", "city": "WAW", "age": 20},
    ]
    out = stable_two_key_sort(
        records,
        primary_key=lambda r: r["city"],
        secondary_key=lambda r: r["age"],
    )
    expected = sorted(records, key=lambda r: (r["city"], r["age"]))
    assert out == expected

_test_stable_two_key_sort()
print("✅ stable_two_key_sort passed")


## 7. Benchmark: gdzie naprawdę widać różnice?

Teraz robimy to, co w praktyce jest najważniejsze po poprawności: **pomiar**.

### Co chcemy zobaczyć?
- **bubble / selection**: wyraźnie przegrywają przy większym `n`,
- **insertion**: zaskakująco dobrze radzi sobie na danych prawie posortowanych,
- **merge**: zachowuje się przewidywalnie na różnych danych,
- **quick**: bywa bardzo szybki, ale prosty wybór pivota może go pogorszyć na niektórych wejściach,
- **`sorted()` / `np.sort()`**: wygrywają w praktyce, bo są dojrzałymi implementacjami bibliotecznymi.

### Ważne
Nie porównujemy tu "czystej teorii", tylko **realne wykonanie w Pythonie**.  
Dlatego niektóre różnice biorą się zarówno ze złożoności, jak i z kosztów implementacyjnych.


In [ ]:
def quicksort_copy(arr: Sequence[int]) -> List[int]:
    a = list(arr)
    quicksort_inplace(a)
    return a

N = 400

datasets = {
    "random": make_random_list(N, seed=0),
    "nearly_sorted": make_nearly_sorted_list(N, swaps=max(1, N // 20), seed=0),
    "reversed": make_reversed_list(N),
}

sorters = {
    "bubble": bubble_sort,
    "selection": selection_sort,
    "insertion": insertion_sort,
    "merge": merge_sort,
    "quick": quicksort_copy,
    "sorted()": sorted,
    "np.sort": lambda xs: np.sort(np.asarray(xs)),
}

rows = benchmark_suite(sorters, datasets, repeats=3)
print_benchmark_table(rows, sort_order=list(sorters.keys()))


### Jak interpretować wyniki?

Najczęściej zobaczysz coś w tym stylu:

- **bubble** i **selection** rosną szybko, bo robią bardzo dużo pracy lokalnej,
- **insertion** na `nearly_sorted` potrafi wypaść dużo lepiej niż na danych losowych,
- **merge** ma bardziej przewidywalne czasy, bo jego logika nie zależy tak mocno od "układu" danych,
- **quick** z prostym pivotem może gorzej wyglądać na danych odwróconych,
- **`sorted()`** zwykle wygrywa, bo używa Timsorta zaimplementowanego niskopoziomowo i adaptacyjnie,
- **`np.sort()`** jest naturalnym narzędziem wtedy, gdy pracujesz na tablicach NumPy, a nie na listach Pythona.

> Najważniejszy wniosek dydaktyczny:  
> **nie każda rekurencja jest szybka i nie każda prostota jest dobra - o wyniku decyduje pomysł algorytmiczny i charakter danych wejściowych.**


## 8. Pomost do praktyki: `sorted()`, `list.sort()`, `np.sort()`, `np.argsort()`

### Python
- `sorted(xs)` zwraca **nową** listę,
- `xs.sort()` sortuje **in-place**,
- oba używają **Timsorta**: stabilnego i adaptacyjnego algorytmu.

### NumPy
- `np.sort(x)` zwraca posortowaną kopię tablicy,
- `x.sort()` sortuje tablicę w miejscu,
- `np.argsort(x)` zwraca **indeksy**, które ustawiają dane w porządku rosnącym.

To jest bardzo przydatne, gdy chcesz:
- posortować kilka powiązanych tablic tym samym porządkiem,
- zachować oryginalne dane i pracować na indeksach.


In [ ]:
x = np.array([7, 2, 5, 2])

idx = np.argsort(x)

print("x             =", x)
print("np.sort(x)    =", np.sort(x))
print("np.argsort(x) =", idx)
print("x[idx]        =", x[idx])


## Podsumowanie / checklista oddania

Po tym notebooku student powinien umieć powiedzieć:

- czym różni się **bubble**, **selection**, **insertion**, **merge** i **quick**,
- które algorytmy są **stabilne**,
- które działają **in-place**,
- dlaczego **merge sort** ma przewidywalne `O(n log n)`,
- dlaczego **quicksort** może być świetny średnio, ale słaby w złym przypadku,
- dlaczego **insertion sort** pomaga na krótkich albo prawie posortowanych fragmentach,
- dlaczego w praktyce używamy najczęściej **`sorted()`**, **`list.sort()`** albo **`np.sort()`**.

### Krótkie pytania refleksyjne
1. Dlaczego selection sort nie jest zwykle stabilny?
2. Dlaczego insertion sort bywa dobry na danych prawie posortowanych?
3. Dlaczego sama rekurencja nie jest źródłem przyspieszenia?
4. Co w benchmarku mówi nam o charakterze danych wejściowych?
